# Tsuki epoch-0 trained Base/full-ICL inference

This notebook is portable across Colab accounts and devices. It downloads the pinned private Hugging Face checkpoint, asks for the reference MP3 and Japanese transcript, and runs the **trained epoch-0 Base/full-ICL checkpoint** through `generate_voice_clone`.

This is not a zero-shot fallback and not a `custom_voice` speaker-ID model. The checkpoint is trained, but its Base/full-ICL inference contract still requires reference audio and reference text. The Hugging Face account used below must have read access to the private repository. No token is stored in this notebook.

## 1. Install the runtime dependencies

Run this cell once after selecting a GPU runtime (T4 or better).

In [ ]:
%pip install -q qwen-tts huggingface_hub soundfile

import importlib.metadata
print('qwen-tts', importlib.metadata.version('qwen-tts'))
print('huggingface-hub', importlib.metadata.version('huggingface-hub'))

## 2. Authenticate to the private Hugging Face repository

Create a Hugging Face **read** token on the account that should run this notebook. Paste it only into the prompt or set the temporary `HF_ACCESS_TOKEN` environment variable; never write it into the notebook.

In [ ]:
import getpass
import os
from huggingface_hub import HfApi, login

access_token = os.environ.get('HF_ACCESS_TOKEN') or getpass.getpass('Hugging Face read token: ')
if not access_token:
    raise RuntimeError('A Hugging Face read token is required for this private repository.')
login(token=access_token, add_to_git_credential=False)

HF_REPO_ID = "tsukihime-exe/tsuki-qwen3-tts-models"
HF_REVISION = "cf726a2d11c565b3a67407fbf91061b13307feb1"
TRAINING_MODE = 'TRAINED_BASE_FULL_ICL'
repo_info = HfApi(token=access_token).model_info(HF_REPO_ID, revision=HF_REVISION)
if not repo_info.private:
    raise RuntimeError('The expected repository is not private.')
print(f'Authenticated for {repo_info.id}; private={repo_info.private}; revision={repo_info.sha}')

## 3. Upload the reference audio and transcript

Select both files when prompted: `tsuki_25s_clip.mp3` and its matching `tsuki_ref_text.txt`. The transcript must describe the audio exactly.

In [ ]:
import hashlib
import os
import shutil
from pathlib import Path

REFERENCE_AUDIO_NAME = 'tsuki_25s_clip.mp3'
REFERENCE_TEXT_NAME = 'tsuki_ref_text.txt'
REFERENCE_AUDIO_SHA256 = 'ce93bb53f197f0683d46ab4ad240e3d434ef6e2b2340190be98dfe382a0f6530'
REFERENCE_TEXT_SHA256 = 'bd3e849d943b80e75f62243c85c3d75dabe2716c2d1b12535b39960d9406949e'
REFERENCE_DIR = Path('/content/reference')
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

automated_reference_dir = os.environ.get('TSUKI_REFERENCE_DIR')
if automated_reference_dir:
    source_dir = Path(automated_reference_dir)
    for expected_name in (REFERENCE_AUDIO_NAME, REFERENCE_TEXT_NAME):
        shutil.copy2(source_dir / expected_name, REFERENCE_DIR / expected_name)
else:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (REFERENCE_DIR / Path(name).name).write_bytes(data)

audio_path = REFERENCE_DIR / REFERENCE_AUDIO_NAME
text_path = REFERENCE_DIR / REFERENCE_TEXT_NAME
if not audio_path.is_file() or not text_path.is_file():
    raise FileNotFoundError('Upload both tsuki_25s_clip.mp3 and tsuki_ref_text.txt.')
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if sha256_file(audio_path) != REFERENCE_AUDIO_SHA256:
    raise ValueError('The uploaded reference audio is not the authorized tsuki_25s_clip.mp3.')
if sha256_file(text_path) != REFERENCE_TEXT_SHA256:
    raise ValueError('The uploaded transcript does not match the authorized reference audio.')
reference_text = text_path.read_text(encoding='utf-8').strip()
print(f'Reference verified: {audio_path.name}, {text_path.name}')

## 4. Download the pinned trained checkpoint from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ROOT = Path(snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='model',
    revision=HF_REVISION,
    allow_patterns=["epoch-0/*"],
    local_dir='/content/tsuki_epoch0_model',
    token=access_token,
))
MODEL_PATH = Path('/content/tsuki_epoch0_model/epoch-0')
required = ['config.json', 'generation_config.json', 'model.safetensors', 'speech_tokenizer/model.safetensors', 'tokenizer_config.json', 'vocab.json']
missing = [name for name in required if not (MODEL_PATH / name).is_file()]
if missing:
    raise FileNotFoundError(f'Model download is incomplete: {missing}')
print(f'Downloaded trained checkpoint: {MODEL_PATH}')

## 5. Load the trained model

This deliberately loads the private `epoch-0` path. It does not load `Qwen/Qwen3-TTS-12Hz-0.6B-Base` as an inference fallback.

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. In Colab select Runtime > Change runtime type > T4 GPU.')
model = Qwen3TTSModel.from_pretrained(
    MODEL_PATH,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation='eager',
)
print(f'Loaded {TRAINING_MODE} checkpoint from {MODEL_PATH}')

## 6. Generate Japanese and English text with the trained checkpoint

The reference prompt is computed once, then reused. The generation calls below are the actual trained-checkpoint inference path.

In [ ]:
import json
import time
import soundfile as sf
from IPython.display import Audio, display

OUTPUT_DIR = Path('/content/tsuki_epoch0_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prompt_started = time.perf_counter()
voice_clone_prompt = model.create_voice_clone_prompt(
    ref_audio=str(audio_path),
    ref_text=reference_text,
    x_vector_only_mode=False,
)
prompt_seconds = time.perf_counter() - prompt_started

tests = [
    ('japanese', 'こんにちは、今夜も君と話せて嬉しいな。', 'Japanese'),
    ('english', 'Hello, I am happy to talk with you tonight.', 'English'),
]
records = []
for name, text, language in tests:
    started = time.perf_counter()
    wavs, sample_rate = model.generate_voice_clone(
        text=text,
        language=language,
        voice_clone_prompt=voice_clone_prompt,
        max_new_tokens=128,
    )
    elapsed = time.perf_counter() - started
    output_path = OUTPUT_DIR / f'epoch0_trained_{name}.wav'
    sf.write(output_path, wavs[0], sample_rate)
    record = {
        'name': name, 'language': language, 'text': text,
        'generation_seconds': elapsed, 'sample_rate': sample_rate,
        'duration_seconds': len(wavs[0]) / sample_rate,
        'path': str(output_path),
    }
    records.append(record)
    print(json.dumps(record, ensure_ascii=False))
    display(Audio(str(output_path)))

metrics = {'training_mode': TRAINING_MODE, 'hf_revision': HF_REVISION, 'prompt_seconds': prompt_seconds, 'tests': records}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Reference prompt: {prompt_seconds:.2f}s')

## 7. Download the outputs

Run this cell once you have listened to the outputs in the notebook.

In [ ]:
if os.environ.get('TSUKI_SKIP_BROWSER_DOWNLOADS') != '1':
    from google.colab import files
    for output_path in sorted(OUTPUT_DIR.glob('*.wav')):
        files.download(str(output_path))
    files.download(str(OUTPUT_DIR / 'metrics.json'))
else:
    print(f'Outputs are ready in {OUTPUT_DIR}; browser downloads skipped.')